## TESTING THE SIMULATION WITH REAL DATA

In this section we will compare the results of our portfolio simulation with the real results we would have got with the real stock market data in the next 30 days.

In [1]:
import sys
import os

parent_dir = os.path.abspath('..')

if parent_dir not in sys.path:
	sys.path.append(parent_dir)

In [2]:
from src.simulation import *
import numpy as np
import pandas as pd
import yfinance as yf

First we download the real stock data of the next 30 trading days of our raw data to make the test. 

In [3]:
days_30 = yf.download(['SAN', 'ITX.MC', 'AIR.PA', 'SIE.DE', 'IBE.MC'], start="2025-01-01", end="2025-01-31", interval='1d', group_by='ticker')

df = pd.DataFrame(days_30)
df = df.filter(like='Close', axis=1) # Keep only the closing prices

[*********************100%***********************]  5 of 5 completed


### Simulation

In [4]:
returns = pd.read_csv('../data/processed/log_returns.csv', index_col=0, header=0)
cov_matrix = pd.read_csv('../data/processed/covariance_matrix.csv', index_col=0, header=0)
cov_matrix = cov_matrix/252

np.random.seed(2026) # For reproducibility

T = 30 # Time horizon in days
M = 10000 # Number of simulations
P = 100000 # Portfolio value in euros - Equally weighted portfolio between the 5 stoks

portfolio_final_values = simulate_portfolio_returns(returns, cov_matrix, T, M, P)

profit = portfolio_final_values - P # Profit distribution

var_95 = np.percentile(profit, 5) # 5th percentile of the profit distribution

print(var_95)

-9086.383566351113


### Real Data

In [5]:
df

Ticker,IBE.MC,ITX.MC,SIE.DE,SAN,AIR.PA
Price,Close,Close,Close,Close,Close
Date,,,,,
2025-01-02,12.584102,47.785370,180.949371,4.331857,153.777924
2025-01-03,12.741633,47.462498,178.710464,4.360995,152.279892
2025-01-06,12.792602,48.355152,184.738297,4.496973,153.297791
2025-01-07,12.704567,48.279179,185.178421,4.555248,152.107040
2025-01-08,12.681402,48.127239,187.187683,4.526111,152.279892
2025-01-09,12.732368,48.526085,188.106232,NaN,150.359360
2025-01-10,12.422282,47.614441,186.096939,4.487259,151.300415
2025-01-13,12.379869,46.446396,184.833969,4.516398,149.264633


In [6]:
first_day = df.iloc[0]
day_30 = df.iloc[-1]

prices = day_30.values/first_day.values

prices = prices*P/returns.shape[1]

final_portfolio_value = np.sum(prices)

profit = final_portfolio_value - P

print(profit)

7209.10515771604


## ROLLING SIMULATION

Now, to make it more realistic, we would do a rolling engine that iterates from the first day of a year to the last day, thus we could wage the real efectiveness of this *Value at Risk* calculator.

In [7]:
M = 10000        # Number of simulations per day
P = 100000       # Starting portfolio value in euros
WINDOW = 252     # 1-year trailing historical window

print("Initiating rolling backtest. Simulating 10,000 scenarios per day...")
backtest_results = rolling_engine(returns, WINDOW, M, P)

total_days_tested = len(backtest_results)
total_breaches = backtest_results['Breach'].sum()
exception_rate = (total_breaches / total_days_tested) * 100

print(f"Total Days Tested: {total_days_tested}")
print(f"Total VaR Breaches: {total_breaches}")
print(f"Exception Rate: {exception_rate:.2f}% (Target: ~5.00%)")


Initiating rolling backtest. Simulating 10,000 scenarios per day...
Total Days Tested: 491
Total VaR Breaches: 28
Exception Rate: 5.70% (Target: ~5.00%)


In [9]:
backtest_results.to_csv('../data/processed/backtest_results.csv')